# Homework 2 — Problem 2:

In [2]:
import time, copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import category_encoders as ce
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
import torch, torch.nn as nn

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cpu"

## 2(a)

In [7]:
df = pd.read_csv("/Users/rohansivakumar/Desktop/AI:ML/DATA/retail.csv")
CATS = ["StockCode", "CustomerID"]
df["StockCode"] = df["StockCode"].astype(str)
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)

n = len(df)
card = {c: df[c].nunique() for c in CATS}
print("rows:", n, "| unique instances:", card)
for c in CATS:
    vc = df[c].value_counts()
    print(f"{c}: median rows/instances = {vc.median():.0f}, instances with < 5 rows = {(vc < 5).mean():.1%}")

train_df, tmp_df = train_test_split(df, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=SEED)
y_tr, y_va, y_te = [np.log1p(d["Quantity"].values) for d in (train_df, val_df, test_df)]

rows: 397924 | unique instances: {'StockCode': 3665, 'CustomerID': 4339}
StockCode: median rows/instances = 49, instances with < 5 rows = 12.9%
CustomerID: median rows/instances = 41, instances with < 5 rows = 5.3%


StockCode has 3,665 unique instances and CustomerID has 4,339 (8,004 total). One-hot encoding seems impractical, and even a sparse one would give a weight to each instance. Note that ~13% of stock codes and ~5% of customers have fewer than 5 training rows, so there is a chance that those weights would be poorly estimated and not reflect the true relationship.

In [13]:
def evaluate(name, y_true, pred):
    err = pred - y_true
    return dict(model=name, RMSE=np.sqrt(np.mean(err ** 2)), MAE=np.mean(np.abs(err)),
                R2=1 - np.sum(err ** 2) / np.sum((y_true - y_true.mean()) ** 2))

preds_test, rows = {}, []
def record(name, pred_te):
    preds_test[name] = pred_te
    rows.append(evaluate(name, y_te, pred_te))

record("baseline: train mean", np.full(len(y_te), y_tr.mean()))

# --- categorical vocabulary from TRAIN only; code 0 = unseen ---
cat_maps = {c: pd.Index(train_df[c].unique()) for c in CATS}
lookup = {c: pd.Series(np.arange(1, len(cat_maps[c]) + 1), index=cat_maps[c]) for c in CATS}
def cat_idx(d):
    # level -> 1..n from the training vocabulary; anything unseen -> 0
    return np.stack([d[c].map(lookup[c]).fillna(0).values for c in CATS], 1).astype(np.int64)
I_tr, I_va, I_te = cat_idx(train_df), cat_idx(val_df), cat_idx(test_df)
n_cats = [len(cat_maps[c]) + 1 for c in CATS]
print("fraction of val / test rows with an unseen StockCode, CustomerID:", (I_va == 0).mean(0), (I_te == 0).mean(0))

class TabNet(nn.Module):
    def __init__(self, n_cats, emb_dims, n_num, hidden=128, drop=0.1, unk_p=0.0):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, d) for n, d in zip(n_cats, emb_dims)])
        self.unk_p = unk_p
        self.mlp = nn.Sequential(nn.Linear(sum(emb_dims) + n_num, hidden), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
    def forward(self, cats, nums):
        if self.training and self.unk_p > 0 and len(self.embs) > 0:
            cats = cats.masked_fill(torch.rand(cats.shape, device=cats.device) < self.unk_p, 0)
        parts = [e(cats[:, i]) for i, e in enumerate(self.embs)]
        if nums.shape[1] > 0: parts.append(nums)
        return self.mlp(torch.cat(parts, 1)).squeeze(-1)

def T(a, dtype=torch.float32): return torch.tensor(a, dtype=dtype, device=device)

def make_data(I, nums, y):
    return (T(I, torch.long), T(nums.reshape(len(I), -1)), T(y))

def train_tab(model, tr, va, epochs=30, bs=1024, lr=2e-3, wd=1e-6, patience=4, loss_fn=None, verbose=False):
    loss_fn = loss_fn or nn.MSELoss()        # alternative: nn.HuberLoss(delta=1.0)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr[2]), device=device)
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            loss = loss_fn(model(tr[0][idx], tr[1][idx]), tr[2][idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(va[0], va[1]), va[2]).item()
        if verbose: print(f"epoch {ep+1:2d} val loss {val:.4f}")
        if val < best - 1e-4: best, best_state, bad = val, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_tab(model, data, bs=8192):
    model.eval()
    return np.concatenate([model(data[0][i:i+bs], data[1][i:i+bs]).cpu().numpy() for i in range(0, len(data[0]), bs)])

fraction of val / test rows with an unseen StockCode, CustomerID: [0.00065339 0.00038533] [0.00050261 0.00040208]


Output above shows that there aren't many instances in which the trained model is tested on something unknown to it.

## 2(b)

In [17]:
SMOOTH = 10  #default

# Target encoding: replace each level with its (smoothed) mean log1p(Quantity) from the TRAIN set
te = ce.TargetEncoder(cols=CATS, smoothing=SMOOTH)
te.fit(train_df[CATS], pd.Series(y_tr, index=train_df.index))
TE_tr = te.transform(train_df[CATS]).values
TE_va = te.transform(val_df[CATS]).values      # unseen levels get the global mean
TE_te = te.transform(test_df[CATS]).values

def standardize(tr, *others):
    m, s = tr.mean(0), tr.std(0) + 1e-8
    return [(a - m) / s for a in (tr, *others)]
    
# MLP Model on the two target-encoded columns
no_cat = lambda N: np.zeros((N, 0), dtype=np.int64)
a, b, c = standardize(TE_tr, TE_va, TE_te)
torch.manual_seed(SEED)
m = TabNet([], [], n_num=2)
m = train_tab(m, make_data(no_cat(len(a)), a, y_tr), make_data(no_cat(len(b)), b, y_va))
record("Target encoding + MLP", predict_tab(m, make_data(no_cat(len(c)), c, y_te)))

display(pd.DataFrame(rows).set_index("model").round(4))

,RMSE,MAE,R2
model,,,
baseline: train mean,1.0072,0.8193,-0.0000
Target encoding + MLP,0.5931,0.4291,0.6532


The MLP model with target encoding shows significant improvement in RMSE, MAE, and $R^2$

## 2(c)

In [20]:
EMB = 16 # looked up "rule of thumb" for selecting reasonable embedding dimension (fourth root rule)
torch.manual_seed(SEED)
model_emb = TabNet(n_cats, [EMB, EMB], n_num=0, unk_p=0.05)
model_emb = train_tab(model_emb, make_data(I_tr, np.zeros((len(I_tr), 0)), y_tr),
                      make_data(I_va, np.zeros((len(I_va), 0)), y_va), verbose=True)
record("Embeddings only", predict_tab(model_emb, make_data(I_te, np.zeros((len(I_te), 0)), y_te)))
display(pd.DataFrame(rows).set_index("model").round(4))

epoch  1 val loss 0.6368
epoch  2 val loss 0.4298
epoch  3 val loss 0.3625
epoch  4 val loss 0.3315
epoch  5 val loss 0.3143
epoch  6 val loss 0.3049
epoch  7 val loss 0.2993
epoch  8 val loss 0.2957
epoch  9 val loss 0.2935
epoch 10 val loss 0.2926
epoch 11 val loss 0.2931
epoch 12 val loss 0.2880
epoch 13 val loss 0.2846
epoch 14 val loss 0.2841
epoch 15 val loss 0.3096
epoch 16 val loss 0.2890
epoch 17 val loss 0.2799
epoch 18 val loss 0.2810
epoch 19 val loss 0.2770
epoch 20 val loss 0.2807
epoch 21 val loss 0.2763
epoch 22 val loss 0.2798
epoch 23 val loss 0.2834
epoch 24 val loss 0.2796
epoch 25 val loss 0.2746
epoch 26 val loss 0.2805
epoch 27 val loss 0.2745
epoch 28 val loss 0.2764
epoch 29 val loss 0.2706
epoch 30 val loss 0.2706


,RMSE,MAE,R2
model,,,
baseline: train mean,1.0072,0.8193,-0.0000
Target encoding + MLP,0.5931,0.4291,0.6532
Embeddings only,0.5286,0.3729,0.7245


We see improvement in RMSE (lower), MAE (lower), and R^2 (larger) when using the embedding model vs the MLP with target encoding. This is evidence that the embedding approach is better than target encoding for this sample (including the seed, split, etc.)

## 2(d)

In [23]:
a, b, c = standardize(TE_tr, TE_va, TE_te)
torch.manual_seed(SEED)
model_comb = TabNet(n_cats, [EMB, EMB], n_num=2, unk_p=0.05)
model_comb = train_tab(model_comb, make_data(I_tr, a, y_tr), make_data(I_va, b, y_va), verbose=True)
record("Embeddings + TE", predict_tab(model_comb, make_data(I_te, c, y_te)))

summary = pd.DataFrame(rows).set_index("model")
display(summary.round(4))

epoch  1 val loss 0.3657
epoch  2 val loss 0.3435
epoch  3 val loss 0.3272
epoch  4 val loss 0.3157
epoch  5 val loss 0.3341
epoch  6 val loss 0.3005
epoch  7 val loss 0.2980
epoch  8 val loss 0.2904
epoch  9 val loss 0.2886
epoch 10 val loss 0.2856
epoch 11 val loss 0.2825
epoch 12 val loss 0.2802
epoch 13 val loss 0.2788
epoch 14 val loss 0.2792
epoch 15 val loss 0.2764
epoch 16 val loss 0.2754
epoch 17 val loss 0.2760
epoch 18 val loss 0.2738
epoch 19 val loss 0.2718
epoch 20 val loss 0.2778
epoch 21 val loss 0.2769
epoch 22 val loss 0.2723
epoch 23 val loss 0.2702
epoch 24 val loss 0.2683
epoch 25 val loss 0.2732
epoch 26 val loss 0.2770
epoch 27 val loss 0.2679
epoch 28 val loss 0.2669
epoch 29 val loss 0.2678
epoch 30 val loss 0.2671


,RMSE,MAE,R2
model,,,
baseline: train mean,1.0072,0.8193,-0.0000
Target encoding + MLP,0.5931,0.4291,0.6532
Embeddings only,0.5286,0.3729,0.7245
Embeddings + TE,0.5247,0.3647,0.7286


We see a slight improvement in both RMSE (lower) and $R^2$ (larger) when using the combined model versus the models that only use target encoding or embeddings only.

## 2(e)

Embedding seems to fix the "high-cardinality" problem. Note that when we were considering using something like one-hot encoding, the resulting vectors would have been too large to put into practice, given how many unique instances there were of each of the two categorical variables. Embedding allows us to reduce that problem by fixing the dimension to 16. Given that target encoding gives each instance one number (the average), the embedding gives a vector that the network can combine across the two features. The model can thus learn about interactions given the hidden/latent features that the embedding layer captures/identifies.